In [ ]:
import gymnasium
!pip install 'mujoco>=2.3.1'
!pip install 'swig'
!pip install highway-env
import highway_env
import numpy
import random
import io
import warnings
import imageio
warnings.filterwarnings('ignore', category=DeprecationWarning, module='imageio')
import IPython
!pip install stable_baselines3
import stable_baselines3
!pip install sb3-contrib
import sb3_contrib
warnings.filterwarnings('ignore', category=DeprecationWarning)
import torch
!pip install imitation
import imitation
!pip install sb3-contrib
import sb3_contrib
import os
!apt-get update -y
!apt-get install -y libglvnd-dev libosmesa6-dev patchelf libglew-dev
!pip install deap
import deap
# Multi-Agent environments
!pip install mpe2
!pip install -q pettingzoo[all]
# Vectorization + wrappers
!pip install -q supersuit
!pip install ray[rllib]

In [ ]:
environment = gymnasium.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')
state_space_size = environment.observation_space.n
action_space_size = environment.action_space.n
q_table_for_q_learning = numpy.zeros((state_space_size, action_space_size))
print('Environment Ready')

In [ ]:
learning_rate = 0.8
discount_factor = 0.95
exploration_epsilon_value = 1.0
exploration_epsilon_minimum_value = 0.05
exploration_epsilon_decay_rate = 0.995
number_of_training_episodes = 5000
maximum_steps_per_episode = 100
list_of_episode_rewards = []
for episode_index in range(number_of_training_episodes):
    current_state, environment_information = environment.reset()
    total_reward_this_episode = 0
    episode_finished = False
    step_counter = 0
    while not episode_finished and step_counter < maximum_steps_per_episode:
        if random.uniform(0, 1) < exploration_epsilon_value:
            chosen_action = environment.action_space.sample()
        else:
            chosen_action = numpy.argmax(q_table_for_q_learning[current_state])
        next_state, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
        episode_finished = episode_terminated or episode_truncated
        q_table_for_q_learning[current_state, chosen_action] = ((1 - learning_rate) * q_table_for_q_learning[current_state, chosen_action] + learning_rate * (current_step_reward + discount_factor * numpy.max(q_table_for_q_learning[next_state])))
        current_state = next_state
        total_reward_this_episode += current_step_reward
        step_counter += 1
    exploration_epsilon_value = max(exploration_epsilon_minimum_value, exploration_epsilon_value * exploration_epsilon_decay_rate)
    list_of_episode_rewards.append(total_reward_this_episode)
print('Training Finished')

In [ ]:
current_state, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < maximum_steps_per_episode:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    chosen_action = numpy.argmax(q_table_for_q_learning[current_state])
    current_state, current_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: FrozenLake-v1')
print('Algorithm: Q-Learning')

In [ ]:
environment = gymnasium.make('Taxi-v3', render_mode='rgb_array')
state_space_size = environment.observation_space.n
action_space_size = environment.action_space.n
q_table_for_sarsa = numpy.zeros((state_space_size, action_space_size))
print('Environment Ready')

In [ ]:
learning_rate = 0.7
discount_factor = 0.95
exploration_epsilon_value = 1.0
exploration_epsilon_minimum_value = 0.05
exploration_epsilon_decay_rate = 0.995
number_of_training_episodes = 5000
maximum_steps_per_episode = 200
list_of_episode_rewards = []
for episode_index in range(number_of_training_episodes):
    current_state, environment_information = environment.reset()
    total_reward_this_episode = 0
    episode_finished = False
    step_counter = 0
    if random.uniform(0, 1) < exploration_epsilon_value:
        chosen_action = environment.action_space.sample()
    else:
        chosen_action = numpy.argmax(q_table_for_sarsa[current_state])
    while not episode_finished and step_counter < maximum_steps_per_episode:
        next_state, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
        episode_finished = episode_terminated or episode_truncated
        if random.uniform(0, 1) < exploration_epsilon_value:
            next_action = environment.action_space.sample()
        else:
            next_action = numpy.argmax(q_table_for_sarsa[next_state])
        q_table_for_sarsa[current_state, chosen_action] = ((1 - learning_rate) * q_table_for_sarsa[current_state, chosen_action] + learning_rate * (current_step_reward + discount_factor * q_table_for_sarsa[next_state, next_action]))
        current_state = next_state
        chosen_action = next_action
        total_reward_this_episode += current_step_reward
        step_counter += 1
    exploration_epsilon_value = max(exploration_epsilon_minimum_value, exploration_epsilon_value * exploration_epsilon_decay_rate)
    list_of_episode_rewards.append(total_reward_this_episode)
print('Training Finished')

In [ ]:
collected_rendered_frames = []
current_state, environment_information = environment.reset()
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < maximum_steps_per_episode:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    chosen_action = numpy.argmax(q_table_for_sarsa[current_state])
    current_state, current_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Taxi-v3')
print('Algorithm: SARSA')

In [ ]:
environment = gymnasium.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')
state_space_size = environment.observation_space.n
action_space_size = environment.action_space.n
state_value_function = numpy.zeros(state_space_size)
print('Environment Ready')

In [ ]:
learning_rate = 0.1
discount_factor = 0.99
number_of_training_episodes = 5000
maximum_steps_per_episode = 100
list_of_episode_rewards = []
state_value_function = numpy.zeros(environment.observation_space.n)
for episode_index in range(number_of_training_episodes):
    current_state, environment_information = environment.reset()
    total_reward_this_episode = 0
    episode_finished = False
    step_counter = 0
    while not episode_finished and step_counter < maximum_steps_per_episode:
        chosen_action = environment.action_space.sample()
        next_state, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
        episode_finished = episode_terminated or episode_truncated
        state_value_function[current_state] = state_value_function[current_state] + learning_rate * (current_step_reward + discount_factor * state_value_function[next_state] - state_value_function[current_state])
        current_state = next_state
        total_reward_this_episode += current_step_reward
        step_counter += 1
    list_of_episode_rewards.append(total_reward_this_episode)
print('Training Finished')

In [ ]:
collected_rendered_frames = []
current_state, environment_information = environment.reset()
episode_done = False
step_counter = 0
while not episode_done and step_counter < maximum_steps_per_episode:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    action_values_for_state = {}
    for action in range(environment.action_space.n):
        transition_information = environment.unwrapped.P[current_state][action][0]
        probability, next_state_from_action, reward_from_action, terminated_from_action = transition_information
        action_value = reward_from_action + discount_factor * state_value_function[next_state_from_action]
        action_values_for_state[action] = action_value
    chosen_action = max(action_values_for_state, key=action_values_for_state.get)
    current_state, current_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
    episode_done = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: FrozenLake-v1')
print('Algorithm: Temporal Difference Learning With A Lookahead Of 0')

In [ ]:
environment = gymnasium.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')
state_space_size = environment.observation_space.n
action_space_size = environment.action_space.n
q_table_for_dyna_q = numpy.zeros((state_space_size, action_space_size))
agent_memory = {}
print('Environment Ready')

In [ ]:
learning_rate = 0.8
discount_factor = 0.95
exploration_epsilon_value = 1.0
exploration_epsilon_minimum_value = 0.05
exploration_epsilon_decay_rate = 0.995
number_of_training_episodes = 50000
maximum_steps_per_episode = 100
number_of_planning_steps = 20
list_of_episode_rewards = []
for episode_index in range(number_of_training_episodes):
    current_state, environment_information = environment.reset()
    total_reward_this_episode = 0
    episode_finished = False
    step_counter = 0
    while not episode_finished and step_counter < maximum_steps_per_episode:
        if random.uniform(0, 1) < exploration_epsilon_value:
            chosen_action = environment.action_space.sample()
        else:
            chosen_action = numpy.argmax(q_table_for_dyna_q[current_state])
        next_state, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
        episode_finished = episode_terminated or episode_truncated
        q_table_for_dyna_q[current_state, chosen_action] = ((1 - learning_rate) * q_table_for_dyna_q[current_state, chosen_action] + learning_rate * (current_step_reward + discount_factor * numpy.max(q_table_for_dyna_q[next_state])))
        agent_memory[(current_state, chosen_action)] = (next_state, current_step_reward)
        for _ in range(number_of_planning_steps):
            if not agent_memory:
                break
            (state_from_memory, action_from_memory), (next_state_from_memory, reward_from_memory) = random.choice(list(agent_memory.items()))
            q_table_for_dyna_q[state_from_memory, action_from_memory] = ((1 - learning_rate) * q_table_for_dyna_q[state_from_memory, action_from_memory] + learning_rate * (reward_from_memory + discount_factor * numpy.max(q_table_for_dyna_q[next_state_from_memory])))
        current_state = next_state
        total_reward_this_episode += current_step_reward
        step_counter += 1
    exploration_epsilon_value = max(exploration_epsilon_minimum_value, exploration_epsilon_value * exploration_epsilon_decay_rate)
    list_of_episode_rewards.append(total_reward_this_episode)
print('Training Finished')

In [ ]:
collected_rendered_frames = []
current_state, environment_information = environment.reset()
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < maximum_steps_per_episode:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    chosen_action = numpy.argmax(q_table_for_dyna_q[current_state])
    current_state, current_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: FrozenLake-v1')
print('Algorithm: Dyna-Q')

In [ ]:
environment = gymnasium.make('CartPole-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.DQN('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=250000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 500:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: CartPole-v1')
print('Algorithm: Deep Q-Network')

In [ ]:
environment = gymnasium.make('LunarLander-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.DQN('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=2500000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 800:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: LunarLander-v3')
print('Algorithm: Double Deep Q-Network')

In [ ]:
environment = gymnasium.make('Acrobot-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.DQN('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=2500000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Acrobot-v1')
print('Algorithm: Dueling DQN')

In [ ]:
environment = gymnasium.make('MountainCar-v0', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = sb3_contrib.QRDQN('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, batch_size=64, buffer_size=100000, learning_starts=1000, target_update_interval=500, verbose=0)
agent.learn(total_timesteps=50000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 500:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: MountainCar-v0')
print('Algorithm: Quantile Regression Replay Buffer With Deep Q-Network')

In [ ]:
environment = gymnasium.make('CartPole-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
class SharedPolicyAndValueNetwork(torch.nn.Module):
    def __init__(self):
        super(SharedPolicyAndValueNetwork, self).__init__()
        # Determine the input size based on the type of observation space
        if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
            obs_input_size = environment.observation_space.n
        elif isinstance(environment.observation_space, gymnasium.spaces.Box):
            obs_input_size = environment.observation_space.shape[0]
        else:
            raise TypeError("Unsupported observation space type.")

        self.shared_layer = torch.nn.Linear(obs_input_size, 128)
        self.policy_layer = torch.nn.Linear(128, environment.action_space.n)
        self.value_layer = torch.nn.Linear(128, 1)

    def forward(self, observation):
        return_observation = observation

        # Handle discrete observation space (one-hot encoding)
        if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
            if observation.dim() == 0:  # Single state index
                return_observation = torch.zeros(environment.observation_space.n, dtype=torch.float32, device=observation.device)
                return_observation[observation] = 1.0
                return_observation = return_observation.unsqueeze(0)
            elif observation.dim() == 1 and observation.dtype == torch.long:  # Batch of state indices
                batch_size = observation.size(0)
                return_observation = torch.zeros(batch_size, environment.observation_space.n, dtype=torch.float32, device=observation.device)
                return_observation.scatter_(1, observation.unsqueeze(1), 1.0)
            else:
                # Assume it's already in the correct format (e.g., batched one-hot or direct discrete state representation)
                # If observation is already a float tensor of appropriate shape (e.g., one-hot), use it directly
                return_observation = observation.float()
        # Handle continuous observation space (Box)
        elif isinstance(environment.observation_space, gymnasium.spaces.Box):
            if observation.dim() == 1: # Single observation (e.g., from env.reset() or env.step())
                return_observation = observation.float().unsqueeze(0)
            elif observation.dim() > 1: # Already batched
                return_observation = observation.float()
            else:
                raise ValueError("Unexpected scalar observation for Box space. Expected a 1D tensor or batch.")
        else:
            raise TypeError("Unsupported observation space type in forward pass.")

        hidden_state = torch.relu(self.shared_layer(return_observation))
        action_probabilities = torch.softmax(self.policy_layer(hidden_state), dim=-1)
        state_value = self.value_layer(hidden_state)

        return action_probabilities, state_value


global_agent = SharedPolicyAndValueNetwork()
optimizer = torch.optim.Adam(global_agent.parameters(), lr=5e-4)

discount_factor_gamma = 0.99
maximum_steps_per_episode = 200


def run_worker_episode(global_agent, worker_environment, discount_factor_gamma, maximum_steps_per_episode):
    episode_probabilities_logarithms = []
    episode_rewards = []
    episode_state_values = []

    current_environment_state_observation, _ = worker_environment.reset()
    episode_finished = False
    step_counter = 0

    while not episode_finished and step_counter < maximum_steps_per_episode:
        # Convert observation to a tensor with the correct dtype based on space type
        if isinstance(worker_environment.observation_space, gymnasium.spaces.Discrete):
            obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.long)
        elif isinstance(worker_environment.observation_space, gymnasium.spaces.Box):
            obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.float32)
        else:
            raise TypeError("Unsupported observation space type for converting to tensor.")

        action_probabilities, state_value_prediction = global_agent(obs_tensor)

        action_distribution = torch.distributions.Categorical(action_probabilities)
        chosen_action = action_distribution.sample()

        next_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, _ = worker_environment.step(chosen_action.item())

        episode_finished = episode_terminated or episode_truncated

        episode_probabilities_logarithms.append(action_distribution.log_prob(chosen_action))
        episode_rewards.append(current_step_reward)
        episode_state_values.append(state_value_prediction.squeeze())

        current_environment_state_observation = next_environment_state_observation
        step_counter += 1

    return episode_probabilities_logarithms, episode_rewards, episode_state_values


if __name__ == '__main__':

    entropy_coefficient = 0.01

    for training in range(100000):

        all_probabilities_logarithms = []
        all_returns = []
        all_estimated_values = []
        all_entropies = []

        worker_results = []

        for worker in range(3):
            episode_probabilities_logarithms, episode_rewards, episode_state_values = run_worker_episode(
                global_agent, environment, discount_factor_gamma, maximum_steps_per_episode
            )
            worker_results.append((episode_probabilities_logarithms, episode_rewards, episode_state_values))
        for episode_probabilities_logarithms, episode_rewards, episode_state_values in worker_results:
            returns = []
            cumulative_reward = 0
            for reward in reversed(episode_rewards):
                cumulative_reward = reward + discount_factor_gamma * cumulative_reward
                returns.insert(0, cumulative_reward)
            if episode_probabilities_logarithms:
                all_probabilities_logarithms.extend(episode_probabilities_logarithms)
                all_returns.extend([torch.tensor(r, dtype=torch.float32) for r in returns])
                all_estimated_values.extend(episode_state_values)
        if all_probabilities_logarithms:
            all_tensor_probabilities_logarithms = torch.stack(all_probabilities_logarithms)
            all_tensor_returns = torch.stack(all_returns)
            all_tensor_estimated_values = torch.stack(all_estimated_values)
            # Normalize returns
            normalized_returns = (all_tensor_returns - all_tensor_returns.mean()) / (all_tensor_returns.std() + 1e-8)
            advantage = normalized_returns - all_tensor_estimated_values.detach()
            # Actor loss
            actor_loss = -(all_tensor_probabilities_logarithms * advantage).mean()
            # Critic loss
            critic_loss = (normalized_returns - all_tensor_estimated_values).pow(2).mean()
            # Entropy bonus (exploration)
            entropy = -all_tensor_probabilities_logarithms.mean()
            total_loss = actor_loss + 0.5 * critic_loss - entropy_coefficient * entropy
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(global_agent.parameters(), 0.5)
            optimizer.step()
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700: # Increased maximum steps for testing
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    # One-hot encode the observation for the global_agent
    # The observation for CartPole-v1 is a Box, not Discrete
    if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
        obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.long)
    elif isinstance(environment.observation_space, gymnasium.spaces.Box):
        obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.float32)
    else:
        raise TypeError("Unsupported observation space type for converting to tensor.")
    action_probabilities, _ = global_agent(obs_tensor)
    optimal_chosen_action = torch.argmax(action_probabilities).item()
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: CartPole-v1')
print('Algorithm: REINFORCE')

In [ ]:
environment = gymnasium.make('LunarLander-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.A2C('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=500000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 800:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: LunarLander-v3')
print('Algorithm: Advantage Actor-Critic')

In [ ]:
environment = gymnasium.make('LunarLander-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
import torch.multiprocessing as mp
# NETWORK
class SharedPolicyAndValueNetwork(torch.nn.Module):
    def __init__(self):
        super(SharedPolicyAndValueNetwork, self).__init__()

        # Determine the input size based on the type of observation space
        if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
            obs_input_size = environment.observation_space.n
        elif isinstance(environment.observation_space, gymnasium.spaces.Box):
            obs_input_size = environment.observation_space.shape[0]
        else:
            raise TypeError("Unsupported observation space type.")

        self.shared_layer = torch.nn.Linear(obs_input_size, 128)
        self.policy_layer = torch.nn.Linear(128, environment.action_space.n)
        self.value_layer = torch.nn.Linear(128, 1)

    def forward(self, observation):
        return_observation = observation

        # Handle discrete observation space (one-hot encoding)
        if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
            if observation.dim() == 0:  # Single state index
                return_observation = torch.zeros(environment.observation_space.n, dtype=torch.float32, device=observation.device)
                return_observation[observation] = 1.0
                return_observation = return_observation.unsqueeze(0)
            elif observation.dim() == 1 and observation.dtype == torch.long:  # Batch of state indices
                batch_size = observation.size(0)
                return_observation = torch.zeros(batch_size, environment.observation_space.n, dtype=torch.float32, device=observation.device)
                return_observation.scatter_(1, observation.unsqueeze(1), 1.0)
            else:
                # Assume it's already in the correct format
                return_observation = observation.float()
        # Handle continuous observation space (Box)
        elif isinstance(environment.observation_space, gymnasium.spaces.Box):
            if observation.dim() == 1: # Single observation (e.g., from env.reset() or env.step())
                return_observation = observation.float().unsqueeze(0)
            elif observation.dim() > 1: # Already batched
                return_observation = observation.float()
            else:
                raise ValueError("Unexpected scalar observation for Box space. Expected a 1D tensor or batch.")
        else:
            raise TypeError("Unsupported observation space type in forward pass.")

        hidden_state = torch.relu(self.shared_layer(return_observation))
        action_logits = self.policy_layer(hidden_state)
        state_value = self.value_layer(hidden_state)

        return action_logits, state_value

# GLOBAL MODEL

global_agent = SharedPolicyAndValueNetwork()
global_agent.share_memory()

optimizer = torch.optim.Adam(global_agent.parameters(), lr=1e-4)

discount_factor_gamma = 0.99
maximum_steps_per_episode = 200
num_workers = 5
total_training_episodes = 100000 # Define total training episodes

# WORKER FUNCTION

def run_worker_episode(worker_id, global_agent, worker_environment, total_episodes_counter):

    while total_episodes_counter.value < total_training_episodes: # Loop until global counter reaches limit

        episode_probabilities_logarithms = []
        episode_rewards = []
        episode_state_values = []

        current_state, _ = worker_environment.reset()
        episode_finished = False
        step_counter = 0

        while not episode_finished and step_counter < maximum_steps_per_episode:

            # Convert observation to a tensor with the correct dtype based on space type
            if isinstance(worker_environment.observation_space, gymnasium.spaces.Discrete):
                state_tensor = torch.tensor(current_state, dtype=torch.long)
            elif isinstance(worker_environment.observation_space, gymnasium.spaces.Box):
                state_tensor = torch.tensor(current_state, dtype=torch.float32)
            else:
                raise TypeError("Unsupported observation space type for converting to tensor.")

            action_logits, state_value = global_agent(state_tensor)

            action_distribution = torch.distributions.Categorical(logits=action_logits)

            chosen_action = action_distribution.sample()

            next_state, reward, terminated, truncated, _ = worker_environment.step(chosen_action.item())

            episode_finished = terminated or truncated

            episode_probabilities_logarithms.append(action_distribution.log_prob(chosen_action))
            episode_rewards.append(reward)
            episode_state_values.append(state_value.squeeze())

            current_state = next_state
            step_counter += 1


        # COMPUTE RETURNS

        returns = []
        cumulative_reward = 0

        for r in reversed(episode_rewards):
            cumulative_reward = r + discount_factor_gamma * cumulative_reward
            returns.insert(0, cumulative_reward)

        returns = torch.tensor(returns, dtype=torch.float32)
        values = torch.stack(episode_state_values)

        # ADVANTAGE

        advantage = returns - values.detach()

        # LOSS

        log_probs = torch.stack(episode_probabilities_logarithms)

        actor_loss = -(log_probs * advantage).mean()
        critic_loss = advantage.pow(2).mean()

        total_loss = actor_loss + 0.5 * critic_loss

        # GLOBAL UPDATE

        optimizer.zero_grad()
        total_loss.backward()

        # push gradients to global model
        for global_param in global_agent.parameters():
            global_param._grad = global_param.grad

        optimizer.step()

        with total_episodes_counter.get_lock():
            total_episodes_counter.value += 1
        if total_episodes_counter.value % 1000 == 0:
            print(f"Worker {worker_id}: Total episodes completed: {total_episodes_counter.value}/{total_training_episodes}")

# MAIN

if __name__ == "__main__":
    # Create a shared counter for total episodes
    total_episodes_counter = mp.Value('i', 0)
    processes = []
    for worker_id in range(num_workers):
        p = mp.Process(target=run_worker_episode, args=(worker_id, global_agent, environment, total_episodes_counter))
        p.start()
        processes.append(p)
    for p in processes:
        p.join()
print("Training Finished")

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 800:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    # Convert observation to a tensor with the correct dtype based on space type
    if isinstance(environment.observation_space, gymnasium.spaces.Discrete):
        obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.long)
    elif isinstance(environment.observation_space, gymnasium.spaces.Box):
        obs_tensor = torch.tensor(current_environment_state_observation, dtype=torch.float32)
    else:
        raise TypeError("Unsupported observation space type for converting to tensor.")
    action_probabilities, estimated_state_value = global_agent(obs_tensor)
    optimal_chosen_action = torch.argmax(action_probabilities).item()
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: LunarLander-v3') # Corrected environment name
print('Algorithm: Asynchronous Advantage Actor-Critic')

In [ ]:
environment = gymnasium.make('CarRacing-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO(policy="CnnPolicy", env=environment, learning_rate=3e-4, gamma=0.99, n_steps=2048, batch_size=64, verbose=0)
agent.learn(total_timesteps=900000)
print("Training Finished")

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: CarRacing-v3')
print('Algorithm: Proximal Policy Optimization')

In [ ]:
environment = gymnasium.make('Acrobot-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = sb3_contrib.TRPO('MlpPolicy', environment, learning_rate=5e-4, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=200000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Acrobot-v1')
print('Algorithm: Trust Region Policy Optimization')

In [ ]:
environment = gymnasium.make('Walker2d-v4', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, gae_lambda=0.95, verbose=0)
agent.learn(total_timesteps=4000000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 400:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Walker2d-v4')
print('Algorithm: Proximal Policy Optimization With Generalized Advantage Estimation')

In [ ]:
environment = gymnasium.make('BipedalWalker-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.SAC('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=700000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: BipedalWalker-v3')
print('Algorithm: Soft Actor-Critic')

In [ ]:
environment = gymnasium.make('Pendulum-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.TD3('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=300000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 300:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Pendulum-v1')
print('Algorithm: Twin Delayed Deep Deterministic Policy Gradient')

In [ ]:
environment = gymnasium.make('Pendulum-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.DDPG('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=700000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 600:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Pendulum-v1')
print('Algorithm: Deep Deterministic Policy Gradient')

In [ ]:
environment = gymnasium.make('Taxi-v3', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
meta_goals = ['Pickup', 'Dropoff']
meta_controller_q_value_table = numpy.zeros((len(meta_goals), 2))
subpolicy_pickup_q_value_table = numpy.zeros((environment.observation_space.n, environment.action_space.n))
subpolicy_dropoff_q_value_table = numpy.zeros((environment.observation_space.n, environment.action_space.n))
learning_rate_alpha = 0.1
discount_factor_gamma = 0.99
current_exploration_epsilon_value = 1.0
minimum_exploration_epsilon_value = 0.05
exploration_epsilon_decay_rate = 0.995
for training in range(5000):
    current_environment_state, environment_information = environment.reset()
    total_reward_this_episode = 0
    episode_finished = False
    step_counter = 0
    while not episode_finished and step_counter < 200:
        if random.uniform(0, 1) < current_exploration_epsilon_value:
            selected_goal_index = random.choice([0, 1])
        else:
            selected_goal_index = numpy.argmax(meta_controller_q_value_table[:, 0])
        selected_goal = meta_goals[selected_goal_index]
        if selected_goal == 'Pickup':
            q_value_reference = subpolicy_pickup_q_value_table
        else:
            q_value_reference = subpolicy_dropoff_q_value_table
        if random.uniform(0, 1) < current_exploration_epsilon_value:
            chosen_action = environment.action_space.sample()
        else:
            chosen_action = numpy.argmax(q_value_reference[current_environment_state])
        next_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
        episode_finished = episode_terminated or episode_truncated
        q_value_reference[current_environment_state, chosen_action] = ((1 - learning_rate_alpha) * q_value_reference[current_environment_state, chosen_action] + learning_rate_alpha * (current_step_reward + discount_factor_gamma * numpy.max(q_value_reference[next_environment_state_observation])))
        current_environment_state = next_environment_state_observation
        total_reward_this_episode += current_step_reward
    current_exploration_epsilon_value = max(minimum_exploration_epsilon_value, current_exploration_epsilon_value * exploration_epsilon_decay_rate)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_render_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 200:
    rendered_frame = environment.render()
    collected_render_frames.append(rendered_frame)
    selected_goal_index = numpy.argmax(meta_controller_q_value_table[:, 0])
    selected_goal = meta_goals[selected_goal_index]
    if selected_goal == 'Pickup':
        q_value_reference = subpolicy_pickup_q_value_table
    else:
        q_value_reference = subpolicy_dropoff_q_value_table
    optimal_chosen_action = numpy.argmax(q_value_reference[current_environment_state_observation])
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_render_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Taxi-v3')
print('Algorithm: Hierarchical Reinforcement Learning')

In [ ]:
vectorized_environment = stable_baselines3.common.env_util.make_vec_env('CartPole-v1', n_envs=1)
print('Environment Ready')

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
import imitation.algorithms.adversarial.gail
# Prepare the environment and expert
# Create the environment with specified max_episode_steps first
expert_env_for_agent_training = gymnasium.make('CartPole-v1', max_episode_steps=1000)
expert_agent = stable_baselines3.PPO('MlpPolicy', expert_env_for_agent_training, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch': [256, 256]}, verbose=0)
expert_agent.learn(total_timesteps=100000)
print('Expert Agent Training Finished')
# Wrap the environment so rollout data is recorded in 'infos'
# Set max_episode_steps for the vectorized environment
expert_environment = DummyVecEnv([lambda: RolloutInfoWrapper(gymnasium.make("CartPole-v1", render_mode='rgb_array', max_episode_steps=1000))])
# Collect trajectories
trajectories = rollout.rollout(expert_agent, expert_environment, rollout.make_sample_until(min_timesteps=None, min_episodes=10000), rng=numpy.random.default_rng())
print('Expert Trajectories Collected')
# Define a reward network for GAIL
reward_net = BasicRewardNet(expert_environment.observation_space, expert_environment.action_space, normalize_input_layer=RunningNorm)
# GAIL Training
# Added allow_variable_horizon=True to handle CartPole's variable episode lengths
imitator_agent = stable_baselines3.PPO('MlpPolicy', expert_environment, policy_kwargs={'net_arch': [256, 256]}, verbose=0)
gail_trainer = imitation.algorithms.adversarial.gail.GAIL(demonstrations=trajectories, demo_batch_size=32, gen_algo=imitator_agent, venv=expert_environment, reward_net=reward_net, allow_variable_horizon=True)
gail_trainer.train(total_timesteps=100000)
print('Imitator Policy Learned via GAIL')

In [ ]:
import cv2
from PIL import Image, ImageDraw, ImageFont
batched_observation = expert_environment.reset()
current_environment_state_observation = batched_observation[0]
environment_information = {}
collected_rendered_frames = []
episode_finished = False
step_counter = 0
total_reward_during_testing = 0 # Initialize total reward
while not episode_finished and step_counter < 1000:
    # Since expert_environment is a DummyVecEnv with n_envs=1,
    # rendered_frame will be a list containing one frame.
    # We extract the single frame from the list.
    rendered_frames_list = expert_environment.render()
    rendered_frame = rendered_frames_list[0] if isinstance(rendered_frames_list, list) else rendered_frames_list
    optimal_chosen_action, _ = imitator_agent.predict(current_environment_state_observation, deterministic=True);
    # Overlay text of the chosen action on the frame
    img = Image.fromarray(rendered_frame)
    draw = ImageDraw.Draw(img)
    try:
        # Try to load a font, otherwise use a default
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 20)
    except IOError:
        font = ImageFont.load_default()
    action_text = f"Action: {optimal_chosen_action}"
    draw.text((10, 10), action_text, (255, 255, 255), font=font) # White text
    rendered_frame_with_text = numpy.array(img)
    collected_rendered_frames.append(rendered_frame_with_text)
    # expert_environment.step expects actions for all environments
    # even if there's only one. Make optimal_chosen_action a batch.
    batched_optimal_chosen_action = numpy.atleast_1d(optimal_chosen_action)
    # VecEnv.step() returns (observations, rewards, dones, infos)
    current_environment_state_observation_batch, current_step_reward_batch, done_batch, environment_information_batch = expert_environment.step(batched_optimal_chosen_action)
    # Extract scalar values from batch if DummyVecEnv has n_envs=1
    current_environment_state_observation = current_environment_state_observation_batch[0]
    current_step_reward = current_step_reward_batch[0]
    episode_finished = done_batch[0] # done_batch combines terminated and truncated
    total_reward_during_testing += current_step_reward # Accumulate reward
    step_counter += 1
print('Testing Finished')
print(f"Total reward during testing: {total_reward_during_testing}") # Print total reward
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 10), loop=0) # Changed duration to 10 FPS and added font for text overlay
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: CartPole-v1')
print('Algorithm: Generative Adversarial Imitation Learning Algorithm')

In [ ]:
environment = gymnasium.make('Acrobot-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
from deap import base, creator, tools
from deap import algorithms as algorithm
number_of_inputs_nodes = environment.observation_space.shape[0]
number_of_outputs_nodes = environment.action_space.n
maximum_allowed_steps_per_episode = 200000
number_of_hidden_nodes = 16
genome_length = (number_of_inputs_nodes * number_of_hidden_nodes + number_of_hidden_nodes) + (number_of_hidden_nodes * number_of_outputs_nodes + number_of_outputs_nodes)
deap.creator.create('FitnessMax', deap.base.Fitness, weights=(1.0,))
deap.creator.create('Individual', list, fitness=deap.creator.FitnessMax)
def calculate_network_output(genome_weights, input_data):
    weights = numpy.array(genome_weights)
    index = 0
    weights_layer_1 = weights[index : index + number_of_inputs_nodes * number_of_hidden_nodes].reshape(number_of_inputs_nodes, number_of_hidden_nodes)
    index += number_of_inputs_nodes * number_of_hidden_nodes
    biases_layer_1 = weights[index : index + number_of_hidden_nodes]
    index += number_of_hidden_nodes
    weights_layer_2 = weights[index : index + number_of_outputs_nodes * number_of_hidden_nodes].reshape(number_of_outputs_nodes, number_of_hidden_nodes)
    biases_layer_2 = weights[-number_of_outputs_nodes:]
    hidden_layer_output = numpy.tanh(numpy.dot(weights_layer_1.T, input_data) + biases_layer_1)
    final_output = numpy.dot(weights_layer_2, hidden_layer_output) + biases_layer_2
    return final_output
def evaluate_genome_fitness(individual_genome):
    current_environment_state_observation, environment_information = environment.reset()
    episode_total_reward = 0
    for _ in range(maximum_allowed_steps_per_episode):
        q_values_network_output = calculate_network_output(individual_genome, current_environment_state_observation)
        optimal_chosen_action = numpy.argmax(q_values_network_output)
        current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
        episode_total_reward += current_step_reward
        if episode_terminated or episode_truncated:
            break
    return episode_total_reward,
toolbox = deap.base.Toolbox()
toolbox.register('generate_weight', numpy.random.uniform, -2.0, 2.0)
toolbox.register('create_individual', deap.tools.initRepeat, deap.creator.Individual, toolbox.generate_weight, n=genome_length)
toolbox.register('create_population', deap.tools.initRepeat, list, toolbox.create_individual)
toolbox.register('evaluate', evaluate_genome_fitness)
toolbox.register('mate', deap.tools.cxOnePoint)
toolbox.register('mutate', deap.tools.mutGaussian, mu=0, sigma=0.5, indpb=0.05)
toolbox.register('select', deap.tools.selTournament, tournsize=3)
initial_population = toolbox.create_population(n=100)
final_population, logbook = algorithm.eaSimple(initial_population, toolbox, cxpb=0.7, mutpb=0.2, ngen=50, stats=None, halloffame=None, verbose=0)
print('Training Finished')

In [ ]:
from deap import tools
best_individual = tools.HallOfFame(1)
best_individual.update(final_population)
winning_individual = best_individual[0]
current_environment_state_observation, environment_information = environment.reset()
collected_render_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 200:
    rendered_frame = environment.render()
    collected_render_frames.append(rendered_frame)
    q_values_network_output = calculate_network_output(winning_individual, current_environment_state_observation)
    optimal_chosen_action = numpy.argmax(q_values_network_output)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_render_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: Acrobot-v1')
print('Algorithm: Genetic Algorithm')

In [ ]:
from mpe2 import simple_spread_v3
import supersuit as ss
from stable_baselines3.common.vec_env import DummyVecEnv
# Multi-Agent Environment (Cooperative)
environment = simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=100, render_mode='rgb_array')
environment = ss.pad_observations_v0(environment)
environment = ss.pad_action_space_v0(environment)
from stable_baselines3.common.vec_env import DummyVecEnv
# Convert the PettingZoo environment to a single-agent Gymnasium environment
environment = ss.pettingzoo_env_to_vec_env_v1(environment)
# Convert the environment to a single-agent wrapper (shared policy trick)
vectorized_environment = ss.concat_vec_envs_v1(environment, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')
print('Environment Ready')

In [ ]:
agents = PPO('MlpPolicy', vectorized_environment, learning_rate=3e-4, gamma=0.99, policy_kwargs={'net_arch': [256, 256]}, verbose=0)
agents.learn(total_timesteps=200000)
print('Training Finished')

In [ ]:
current_environment_state_observation = vectorized_environment.reset()
collected_rendered_frames = []
episodes_done = False
step_counter = 0
while not episodes_done and step_counter < 200:
    frames = vectorized_environment.render()
    if frames is not None:
        collected_rendered_frames.append(frames)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    action_for_step = numpy.atleast_1d(optimal_chosen_action)
    # VecEnv step returns 4 values: obs, rewards, dones, infos
    current_environment_state_observation, current_step_reward, dones, environment_information = vectorized_environment.step(action_for_step)
    episodes_done = dones.any()
    step_counter += 1
print('Testing Finished')
if collected_rendered_frames:
    gif_buffer_memory = io.BytesIO()
    imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 10), loop=0)
    gif_buffer_memory.seek(0)
    IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: simple_spread_v3')
print('Algorithm: Cooperative Agent Reinforcement Learning')

In [ ]:
environment = gymnasium.make('highway-v0', render_mode='rgb_array')
environment.unwrapped.config['controlled_vehicles'] = 15
environment.unwrapped.config['observation'] = {'type': 'MultiAgentObservation', 'observation_config': {'type': 'Kinematics'}}
environment.unwrapped.config['action'] = {'type': 'MultiAgentAction', 'action_config': {'type': 'DiscreteMetaAction'}}
environment.unwrapped.config['collision_reward'] = -5
environment.unwrapped.config['high_speed_reward'] = 0.5
environment.unwrapped.config['right_lane_reward'] = 0.1
environment.unwrapped.config['offroad_terminal'] = True
environment.unwrapped.config['reward_weights'] = [0.6, 0.3, 0.1]
environment.unwrapped.config['vehicles_count'] = 15
environment.unwrapped.config['duration'] = 40
environment.unwrapped.config['policy_frequency'] = 15
environment.reset()
print('Environment Ready')

In [ ]:
# Create a single-agent environment for training PPO
single_agent_config_ppo = environment.unwrapped.config.copy()
single_agent_config_ppo['controlled_vehicles'] = 1
single_agent_config_ppo['observation'] = {'type': 'Kinematics'}
single_agent_config_ppo['action'] = {'type': 'DiscreteMetaAction'}
single_agent_env_ppo = gymnasium.make('highway-v0', config=single_agent_config_ppo, render_mode='rgb_array')
single_agent_env_ppo.reset()

agent_1 = stable_baselines3.PPO('MlpPolicy', single_agent_env_ppo, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_1.learn(total_timesteps=75000)

# Create a single-agent environment for training TD3
single_agent_config_td3 = environment.unwrapped.config.copy()
single_agent_config_td3['controlled_vehicles'] = 1
single_agent_config_td3['observation'] = {'type': 'Kinematics'}
single_agent_config_td3['action'] = {'type': 'ContinuousAction'}
single_agent_env_td3 = gymnasium.make('highway-v0', config=single_agent_config_td3, render_mode='rgb_array')
single_agent_env_td3.reset()

agent_2 = stable_baselines3.TD3('MlpPolicy', single_agent_env_td3, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_2.learn(total_timesteps=75000)
print('Training Finished')

single_agent_env_ppo.close()
single_agent_env_td3.close()

In [ ]:
# Ensure environment is configured for Multi-Agent
environment.unwrapped.config.update({"observation": {"type": "MultiAgentObservation", "observation_config": {"type": "Kinematics"}}, "action": { "type": "MultiAgentAction", "action_config": {"type": "DiscreteMetaAction"}}, "policy_frequency": 15})
num_controlled_vehicles = environment.unwrapped.config['controlled_vehicles']
# Reset returns a tuple of observations (one per vehicle)
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episodes_done = False
step_counter = 0
while not episodes_done and step_counter < 750:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    actions = []
    # Safety check: ensure we have as many observations as vehicles
    current_obs_len = len(current_environment_state_observation)
    for i in range(num_controlled_vehicles):
        # Use modulo if observations are fewer than expected to avoid IndexError
        obs_i = current_environment_state_observation[i % current_obs_len]
        if i < (num_controlled_vehicles // 2):
            action_i, _ = agent_1.predict(obs_i, deterministic=True)
            actions.append(int(numpy.array(action_i).item()))
        else:
            action_i, _ = agent_2.predict(obs_i, deterministic=True)
            # TD3 outputs continuous actions, need to discretize for DiscreteMetaAction
            continuous_action = action_i.flatten() # Ensure it's a 1D array
            steering_action = continuous_action[0]
            acceleration_action = continuous_action[1]

            # Heuristic mapping from continuous [steering, acceleration] to discrete actions:
            # 0: LANE_LEFT, 1: IDLE, 2: LANE_RIGHT, 3: FASTER, 4: SLOWER
            if steering_action < -0.3:
                chosen_discrete_action = 0 # LANE_LEFT
            elif steering_action > 0.3:
                chosen_discrete_action = 2 # LANE_RIGHT
            elif acceleration_action > 0.5:
                chosen_discrete_action = 3 # FASTER
            elif acceleration_action < -0.5:
                chosen_discrete_action = 4 # SLOWER
            else:
                chosen_discrete_action = 1 # IDLE
            actions.append(chosen_discrete_action)
    try:
        # Step expects a tuple of actions for MultiAgentAction
        current_environment_state_observation, rewards, terminated, truncated, info = environment.step(tuple(actions))
        episodes_done = terminated or truncated
    except Exception as e:
        print(f"Error during execution: {e}")
        break
    step_counter += 1
print('Testing Finished')
if collected_rendered_frames:
    gif_buffer_memory = io.BytesIO()
    imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 20), loop=0)
    gif_buffer_memory.seek(0)
    IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: highway-v0')
print('Algorithm: Cooperative Multi-Agent')

In [ ]:
import ray
import os
# This is often a warning and not critical, but good practice to handle.
if "XDG_RUNTIME_DIR" not in os.environ:
    os.environ["XDG_RUNTIME_DIR"] = "/tmp/ray"
from ray.tune.registry import register_env
from ray.rllib.env.wrappers.pettingzoo_env import PettingZooEnv
from mpe2 import simple_tag_v3

def create_competitive_multi_agent_environment(environment_configuration):
    # Create the raw PettingZoo parallel environment
    env = simple_tag_v3.parallel_env(num_good=1, num_adversaries=3, num_obstacles=2, max_cycles=200, continuous_actions=False)
    # Wrap the PettingZoo environment with RLlib's PettingZooEnv wrapper
    # This wrapper makes the PettingZoo environment compatible with RLlib's GymnasiumEnv expectations.
    return PettingZooEnv(env)

register_env('CompetitiveMultiAgentEnvironment', create_competitive_multi_agent_environment)

# Create a temporary environment to infer observation and action spaces for policy definitions
# This temporary env will now correctly use the PettingZooEnv wrapper
temporary_environment_instance = create_competitive_multi_agent_environment({})

print('Environment Ready')
print('Agents:')

# Access possible_agents from the unwrapped PettingZoo environment within the wrapper
for current_agent_identifier in temporary_environment_instance.unwrapped.possible_agents:
    print(current_agent_identifier)

temporary_environment_instance.close()
ray.shutdown()
ray.init(ignore_reinit_error=True)
print('Ray Ready')

In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.tune.registry import register_env
from ray.rllib.env.wrappers.pettingzoo_env import PettingZooEnv
from mpe2 import simple_tag_v3
from pettingzoo.utils.conversions import parallel_to_aec

# Re-define the environment creation function within this cell's context
def create_competitive_multi_agent_environment(environment_configuration):
    # Create the raw PettingZoo parallel environment, specifying render_mode here
    env = simple_tag_v3.parallel_env(num_good=1, num_adversaries=3, num_obstacles=2, max_cycles=200, continuous_actions=False, render_mode='rgb_array' # <--- Added render_mode here)
    # Convert the parallel environment to an AEC environment
    aec_env = parallel_to_aec(env)
    # Wrap the AEC environment with RLlib's PettingZooEnv wrapper
    # This ensures the environment has the 'agent_selection' attribute expected by RLlib's wrapper.
    return PettingZooEnv(aec_env)

# Re-register the environment after ray.init() has been called (from previous cell)
register_env('CompetitiveMultiAgentEnvironment', create_competitive_multi_agent_environment)

# Re-create the temporary environment instance to infer observation and action spaces
temporary_environment_instance = create_competitive_multi_agent_environment({})

# select agents
policies = {"shared_adversary_policy": (None, temporary_environment_instance.observation_space["adversary_0"], temporary_environment_instance.action_space["adversary_0"], {}), "shared_agent_policy": (None, temporary_environment_instance.observation_space["agent_0"], temporary_environment_instance.action_space["agent_0"], {})}

def policy_mapping_function(agent_identifier, episode, **kwargs):
    if "adversary" in agent_identifier:
        return "shared_adversary_policy"
    else:
        return "shared_agent_policy"

competitive_algorithm = (
    PPOConfig()
    .environment(
        env="CompetitiveMultiAgentEnvironment"
    )
.framework("torch")
    .env_runners(num_env_runners=1)
    .multi_agent(
        policies=policies,
        policy_mapping_fn=policy_mapping_function
    )
    .training(
        lr=3e-4,
        gamma=0.99,
        train_batch_size=14000
    )
    .build()
)

print("Training Initialized")

for training_iteration in range(101):
    training_result = competitive_algorithm.train()
    if training_iteration % 10 == 0:
        print("Iteration:", training_iteration)
print("Training Finished")

In [ ]:
# create test environment
test_environment = create_competitive_multi_agent_environment({})
observations, _ = test_environment.reset()
collected_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 200:
    actions = {}
    for agent_id, observation in observations.items():
        current_policy_id = "shared_adversary_policy" if "adversary" in agent_id else "shared_agent_policy"
        # Get the policy module (RLModule) directly from the algorithm
        policy_module = competitive_algorithm.get_module(current_policy_id)

        # Prepare the observation for inference
        # RLModules expect torch.Tensors as input, usually batched.
        # simple_tag_v3 uses Box observation space (16 for adversary, 14 for agent)
        # so the observation is a numpy array of floats.
        obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0) # Add batch dimension

        # Perform inference to get action logits/distribution parameters
        # The output key for action distribution inputs is typically 'action_dist_inputs'
        inference_output = policy_module.forward_inference({"obs": obs_tensor})
        action_logits = inference_output["action_dist_inputs"]

        # For deterministic action (as is common in testing/evaluation),
        # take the argmax of the logits for discrete action spaces.
        action = torch.argmax(action_logits, dim=-1).squeeze(0).item()
        actions[agent_id] = action

    observations, rewards, terminations, truncations, infos = test_environment.step(actions)

    # render - Directly call render on the wrapped env, which expects no arguments after init
    frame = test_environment.env.render() # <--- Modified here

    if frame is not None:
        collected_frames.append(frame)

    # check end
    episode_finished = all(terminations.values()) or all(truncations.values())
    step_counter += 1

print("Testing Finished")
gif_buffer = io.BytesIO()
imageio.mimsave(gif_buffer, [np.array(frame) for frame in collected_frames], format="GIF", duration=80, loop=0)
gif_buffer.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer.read()))
print('Environment: simple_spread_v3')
print('Algorithm: Competitive Multi-Agent')

In [ ]:
environment = gymnasium.make('highway-v0', render_mode='rgb_array')
environment.unwrapped.config['collision_reward'] = -5
environment.unwrapped.config['controlled_vehicles'] = 10
environment.unwrapped.config['high_speed_reward'] = 0.5
environment.unwrapped.config['right_lane_reward'] = 0.1
environment.unwrapped.config['offroad_terminal'] = True
environment.unwrapped.config['reward_weights'] = [0.6, 0.3, 0.1]
environment.unwrapped.config['vehicles_count'] = 1
environment.unwrapped.config['duration'] = 40
environment.unwrapped.config['policy_frequency'] = 15
environment.unwrapped.config['observation'] = {'type': 'Kinematics'}
environment.reset()
print('Environment Ready')

In [ ]:
agent_1 = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_1.learn(total_timesteps=75000)
agent_2 = sb3_contrib.TRPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_2.learn(total_timesteps=75000)
print('Training Finished')

In [ ]:
# Ensure environment is configured for Multi-Agent
environment.unwrapped.config.update({"observation": {"type": "MultiAgentObservation", "observation_config": {"type": "Kinematics"}}, "action": { "type": "MultiAgentAction", "action_config": {"type": "DiscreteMetaAction"}}, "policy_frequency": 15})
num_controlled_vehicles = environment.unwrapped.config['controlled_vehicles']
# Reset returns a tuple of observations (one per vehicle)
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episodes_done = False
step_counter = 0
while not episodes_done and step_counter < 750:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    actions = []
    # Safety check: ensure we have as many observations as vehicles
    current_obs_len = len(current_environment_state_observation)
    for i in range(num_controlled_vehicles):
        # Use modulo if observations are fewer than expected to avoid IndexError
        obs_i = current_environment_state_observation[i % current_obs_len]
        if i < (num_controlled_vehicles // 2):
            action_i, _ = agent_1.predict(obs_i, deterministic=True)
        else:
            action_i, _ = agent_2.predict(obs_i, deterministic=True)
        actions.append(int(numpy.array(action_i).item()))
    try:
        # Step expects a tuple of actions for MultiAgentAction
        current_environment_state_observation, rewards, terminated, truncated, info = environment.step(tuple(actions))
        episodes_done = terminated or truncated
    except Exception as e:
        print(f"Error during execution: {e}")
        break
    step_counter += 1
print('Testing Finished')
if collected_rendered_frames:
    gif_buffer_memory = io.BytesIO()
    imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 20), loop=0)
    gif_buffer_memory.seek(0)
    IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: highway-v0')
print('Algorithm: Competitive Multi-Agent')

In [ ]:
environment = gymnasium.make('roundabout-v0', render_mode='rgb_array')
environment.unwrapped.config['collision_reward'] = -2
environment.unwrapped.config['controlled_vehicles'] = 4
environment.unwrapped.config['high_speed_reward'] = 0.3
environment.unwrapped.config['right_lane_reward'] = 0.2
environment.unwrapped.config['offroad_terminal'] = True
environment.unwrapped.config['reward_weights'] = [0.5, 0.3, 0.2]
environment.unwrapped.config['vehicles_count'] = 1
environment.unwrapped.config['duration'] = 40
environment.unwrapped.config['policy_frequency'] = 15
environment.unwrapped.config['observation'] = {'type': 'Kinematics'}
environment.reset()
print('Environment Ready')

In [ ]:
agent_1 = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_1.learn(total_timesteps=25000)
agent_2 = stable_baselines3.A2C('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_2.learn(total_timesteps=25000)
print('Training Finished')

In [ ]:
# Ensure environment is configured for Multi-Agent
environment.unwrapped.config.update({"observation": {"type": "MultiAgentObservation", "observation_config": {"type": "Kinematics"}}, "action": { "type": "MultiAgentAction", "action_config": {"type": "DiscreteMetaAction"}}, "policy_frequency": 15})
num_controlled_vehicles = environment.unwrapped.config['controlled_vehicles']
# Reset returns a tuple of observations (one per vehicle)
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episodes_done = False
step_counter = 0
while not episodes_done and step_counter < 100:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    actions = []
    # Safety check: ensure we have as many observations as vehicles
    current_obs_len = len(current_environment_state_observation)
    for i in range(num_controlled_vehicles):
        # Use modulo if observations are fewer than expected to avoid IndexError
        obs_i = current_environment_state_observation[i % current_obs_len]
        if i < (num_controlled_vehicles // 2):
            action_i, _ = agent_1.predict(obs_i, deterministic=True)
        else:
            action_i, _ = agent_2.predict(obs_i, deterministic=True)
        actions.append(int(numpy.array(action_i).item()))
    try:
        # Step expects a tuple of actions for MultiAgentAction
        current_environment_state_observation, rewards, terminated, truncated, info = environment.step(tuple(actions))
        episodes_done = terminated or truncated
    except Exception as e:
        print(f"Error during execution: {e}")
        break
    step_counter += 1

print('Testing Finished')
if collected_rendered_frames:
    gif_buffer_memory = io.BytesIO()
    imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 20), loop=0)
    gif_buffer_memory.seek(0)
    IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: roundabout-v0')
print('Algorithm: Mixed Multi-Agent')

In [ ]:
environment = gymnasium.make('highway-v0', render_mode='rgb_array')
environment.unwrapped.config['collision_reward'] = -2
environment.unwrapped.config['controlled_vehicles'] = 4
environment.unwrapped.config['high_speed_reward'] = 0.3
environment.unwrapped.config['right_lane_reward'] = 0.2
environment.unwrapped.config['offroad_terminal'] = True
environment.unwrapped.config['reward_weights'] = [0.5, 0.3, 0.2]
environment.unwrapped.config['vehicles_count'] = 1
environment.unwrapped.config['duration'] = 40
environment.unwrapped.config['policy_frequency'] = 15
environment.unwrapped.config['observation'] = {'type': 'Kinematics'}
environment.reset()
print('Environment Ready')

In [ ]:
agent_1 = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_1.learn(total_timesteps=75000)
agent_2 = stable_baselines3.A2C('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent_2.learn(total_timesteps=75000)
print('Training Finished')

In [ ]:
# Ensure environment is configured for Multi-Agent
environment.unwrapped.config.update({"observation": {"type": "MultiAgentObservation", "observation_config": {"type": "Kinematics"}}, "action": { "type": "MultiAgentAction", "action_config": {"type": "DiscreteMetaAction"}}, "policy_frequency": 15})
num_controlled_vehicles = environment.unwrapped.config['controlled_vehicles']
# Reset returns a tuple of observations (one per vehicle)
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episodes_done = False
step_counter = 0
while not episodes_done and step_counter < 750:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    actions = []
    # Safety check: ensure we have as many observations as vehicles
    current_obs_len = len(current_environment_state_observation)
    for i in range(num_controlled_vehicles):
        # Use modulo if observations are fewer than expected to avoid IndexError
        obs_i = current_environment_state_observation[i % current_obs_len]
        if i < (num_controlled_vehicles // 2):
            action_i, _ = agent_1.predict(obs_i, deterministic=True)
        else:
            action_i, _ = agent_2.predict(obs_i, deterministic=True)
        actions.append(int(numpy.array(action_i).item()))
    try:
        # Step expects a tuple of actions for MultiAgentAction
        current_environment_state_observation, rewards, terminated, truncated, info = environment.step(tuple(actions))
        episodes_done = terminated or truncated
    except Exception as e:
        print(f"Error during execution: {e}")
        break
    step_counter += 1

print('Testing Finished')
if collected_rendered_frames:
    gif_buffer_memory = io.BytesIO()
    imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 20), loop=0)
    gif_buffer_memory.seek(0)
    IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: highway-v0')
print('Algorithm: Mixed Multi-Agent')

In [ ]:
environment = gymnasium.make('highway-v0', render_mode='rgb_array')
environment.unwrapped.config['collision_reward'] = -5
environment.unwrapped.config['high_speed_reward'] = 0.1
environment.unwrapped.config['right_lane_reward'] = 0.3
environment.unwrapped.config['policy_frequency'] = 15
environment.reset()
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
agent.learn(total_timesteps=20000)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: highway-v0')
print('Algorithm: Safe Reinforcement Learning')

In [ ]:
environment = gymnasium.make('CartPole-v1', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
behavior_policy_agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
behavior_policy_agent.learn(total_timesteps=5000)
current_environment_state_observation, environment_information = environment.reset()
collected_dataset = []
for collecting_step in range(1000):
    chosen_action = environment.action_space.sample()
    next_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(chosen_action)
    episode_finished = episode_terminated or episode_truncated
    collected_dataset.append((current_environment_state_observation, chosen_action, current_step_reward, next_environment_state_observation, episode_finished))
    current_environment_state_observation = next_environment_state_observation if not episode_finished else environment.reset()[0]
class OfflineAgent:
    def __init__(self, environment, lr=1e-3):
        self.environment = environment
        self.gamma = 0.99
        self.q_values_network = torch.nn.Sequential(torch.nn.Linear(environment.observation_space.shape[0],128), torch.nn.ReLU(), torch.nn.Linear(128, environment.action_space.n))
        self.loss_function = torch.nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.q_values_network.parameters(), lr=lr)
    def train(self, collected_dataset):
        for offline_training in range(1000):
            numpy.random.shuffle(collected_dataset)
            for current_environment_state_observation, chosen_action, current_step_reward, next_environment_state_observation, episode_finished in collected_dataset:
                predicted_q_values = self.q_values_network(torch.tensor(current_environment_state_observation, dtype=torch.float32))
                with torch.no_grad():
                    target_q_value = current_step_reward + self.gamma * self.q_values_network(torch.tensor(next_environment_state_observation, dtype=torch.float32)).max() * (1 - episode_finished)
                loss = self.loss_function(predicted_q_values[chosen_action], target_q_value)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
offline_agent = OfflineAgent(environment)
offline_agent.train(collected_dataset)
print('Traning Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_render_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 700:
    rendered_frame = environment.render()
    collected_render_frames.append(rendered_frame)
    predicted_q_values = torch.tensor(current_environment_state_observation, dtype=torch.float)
    optimal_chosen_action_index = torch.argmax(offline_agent.q_values_network(predicted_q_values)).item()
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action_index)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_data = io.BytesIO()
imageio.mimsave(gif_buffer_data, [numpy.array(frame) for frame in collected_render_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_data.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_data.read()))
print('Environment: CartPole‑v1')
print('Algorithm: Offline Reinforcement Learning')

In [ ]:
environment = gymnasium.make('MountainCar-v0', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
for batch_traning in range(10):
    agent.learn(total_timesteps=500000, reset_num_timesteps=False)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 500:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: MountainCar-v0')
print('Algorithm: Batch Reinforcement Learning')

In [ ]:
environment = gymnasium.make('MountainCar-v0', render_mode='rgb_array')
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
for adaptation_training in range(5):
    environment.reset(options={'g': random.uniform(8.5, 10.0)})
    agent.learn(total_timesteps=50000, reset_num_timesteps=False)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 600:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: MountainCar-v0')
print('Algorithm: Meta Reinforcement Learning')

In [ ]:
environment = gymnasium.make('highway-v0', render_mode='rgb_array')
environment.unwrapped.config['collision_reward'] = -5
environment.unwrapped.config['high_speed_reward'] = 0.1
environment.unwrapped.config['right_lane_reward'] = 0.3
environment.unwrapped.config['policy_frequency'] = 15
environment.reset()
print('Environment Ready')

In [ ]:
agent = stable_baselines3.PPO('MlpPolicy', environment, learning_rate=1e-3, gamma=0.99, policy_kwargs={'net_arch':[256, 256]}, verbose=0)
for adaptation_training in range(3):
    environment.reset(options={'g': random.uniform(8.5, 10.0)})
    agent.learn(total_timesteps=25000, reset_num_timesteps=False)
print('Training Finished')

In [ ]:
current_environment_state_observation, environment_information = environment.reset()
collected_rendered_frames = []
episode_finished = False
step_counter = 0
while not episode_finished and step_counter < 600:
    rendered_frame = environment.render()
    collected_rendered_frames.append(rendered_frame)
    optimal_chosen_action, _ = agent.predict(current_environment_state_observation, deterministic=True)
    current_environment_state_observation, current_step_reward, episode_terminated, episode_truncated, environment_information = environment.step(optimal_chosen_action)
    episode_finished = episode_terminated or episode_truncated
    step_counter += 1
print('Testing Finished')
gif_buffer_memory = io.BytesIO()
imageio.mimsave(gif_buffer_memory, [numpy.array(frame) for frame in collected_rendered_frames], format='GIF', duration=int(1000 / 120), loop=0)
gif_buffer_memory.seek(0)
IPython.display.display(IPython.display.Image(data=gif_buffer_memory.read()))
print('Environment: highway-v0')
print('Algorithm: Meta Reinforcement Learning')